### acceptance functions

proportional: proposed maps are accepted with probability: $P = \min(1, \exp(\beta \cdot \Delta TCP)) $
where $\Delta TCP = TCP_{proposed} - TCP_{current}$.

- **proportional** ($\beta \in \{50, 100, 200\}$): exponential decay on bad moves. higher $\beta$ = stricter.
- **margin** (e.g. `margin_10_40`): step-function thresholds
- **simple** (e.g. `simple_25`, `simple_75`): raw tcp threshold acceptance


## Init

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os
import re

sns.set_theme(style='whitegrid')


In [ ]:
### load and aggregate all experiment csvs
RESULTS_DIR = 'results'
TCP_COL = 'unweighted_tcp_score'

csv_files = sorted(glob.glob(f'{RESULTS_DIR}/*/*.csv'))
print(f'found {len(csv_files)} result files')

### known strategies - order matters for matching multi-token names first
STRATEGY_LIST = [
    'proportional_50', 'proportional_100', 'proportional_200',
    'margin_10_40', 'margin_20_50', 'margin_30_60',
    'simple_25', 'simple_50', 'simple_75',
]

CLASS_MAP = {
    'proportional': 'Proportional',
    'margin': 'Margin',
    'simple': 'Simple',
}

summary_rows = []
trace_frames = []  # flat list, no dict key collision

for fpath in csv_files:
    df = pd.read_csv(fpath)
    basename = os.path.basename(fpath)

    ### parse surcharges from directory name
    m = re.search(r'county_(\d+)_coi_(\d+)', fpath)
    county_surcharge = int(m.group(1)) if m else 0
    coi_surcharge = int(m.group(2)) if m else 100

    ### match strategy name
    strategy = 'unknown'
    strategy_class = 'Other'
    for s in STRATEGY_LIST:
        if s in basename:
            strategy = s
            strategy_class = CLASS_MAP[s.split('_')[0]]
            break

    ### surcharge label for plotting (e.g. "county 0 / coi 100")
    surcharge_label = f'county {county_surcharge} / coi {coi_surcharge}'

    summary_rows.append({
        'strategy': strategy,
        'strategy_class': strategy_class,
        'county_surcharge': county_surcharge,
        'coi_surcharge': coi_surcharge,
        'surcharge_label': surcharge_label,
        'mean_tcp': df[TCP_COL].mean(),
        'max_tcp': df[TCP_COL].max(),
        'min_tcp': df[TCP_COL].min(),
        'mean_county_splits': df['county_split_count'].mean(),
        'mean_accept_rate': df['rolling_accept_rate'].mean() if 'rolling_accept_rate' in df.columns else np.nan,
    })

    ### thin trace to every 100th row for plotting
    t = df[['step', TCP_COL, 'county_split_count', 'rolling_accept_rate']].iloc[::100].copy()
    t['strategy'] = strategy
    t['strategy_class'] = strategy_class
    t['county_surcharge'] = county_surcharge
    t['coi_surcharge'] = coi_surcharge
    t['surcharge_label'] = surcharge_label
    trace_frames.append(t)

master_df = pd.DataFrame(summary_rows)
traces = pd.concat(trace_frames, ignore_index=True)

### enforce ordering
CLASS_ORDER = ['Proportional', 'Margin', 'Simple']
traces['strategy_class'] = pd.Categorical(traces['strategy_class'], categories=CLASS_ORDER, ordered=True)

surcharge_order = sorted(traces['surcharge_label'].unique())
traces['surcharge_label'] = pd.Categorical(traces['surcharge_label'], categories=surcharge_order, ordered=True)

print(f'loaded {len(master_df)} runs across {traces["surcharge_label"].nunique()} surcharge configs')
display(master_df.head(10))


## Pareto Frontier: TCP vs County Splits

tradeoff: we want to maximize coi preservation (tcp) while minimizing county splits. so, the ideal maps are sit in the **top-left corner of the plots.**


In [ ]:
g = sns.displot(
    data=traces,
    x="county_split_count",
    y=TCP_COL,
    row="surcharge_label",
    col="strategy",
    kind="kde",
    fill=True,
    alpha=0.7,
    cmap="Blues",
    linewidths=0,
    levels=10,
    thresh=0.05,
    height=3,
    aspect=1
)

g.set_axis_labels("county splits", "tcp score")
g.set_titles(row_template="{row_name}", col_template="{col_name}")
plt.subplots_adjust(top=0.9)
g.fig.suptitle("Pareto Frontier: TCP vs County Splits (top-left is best)", fontsize=16)
plt.show()

## TCP Over Steps

how does tcp evolve as the chain runs? smooth, stabilizing climbs suggest
realistic optimization. erratic swings suggest the acceptance function is
too loose or too tight.




In [ ]:
### tcp trajectory by strategy, faceted by surcharge config
grouped_traces = traces.groupby(['step', 'surcharge_label', 'strategy'], observed=True).mean(numeric_only=True).reset_index()

g = sns.relplot(
    data=grouped_traces,
    x='step',
    y=TCP_COL,
    hue='strategy',
    col='surcharge_label',
    col_wrap=2,
    kind='line',
    height=4,
    aspect=1.5,
    linewidth=2
)

g.set_axis_labels('Chain Step', 'Unweighted TCP Score')
g.set_titles('{col_name}')
g.fig.suptitle('TCP Trajectory Over Chain Steps (By Surcharge Parameter)', y=1.05, fontsize=16)
plt.show()


In [ ]:
# filter based on the top 3 we think we like
top_strategies = ['margin_10_40', 'proportional_100', 'simple_50']
filtered_traces = traces[traces['strategy'].isin(top_strategies)].copy()

grouped_traces = filtered_traces.groupby(['step', 'surcharge_label', 'strategy'], observed=True).mean(numeric_only=True).reset_index()

# 3. Plot with thicker lines and a distinct color palette
g = sns.relplot(
    data=grouped_traces,
    x='step',
    y=TCP_COL,
    hue='strategy',
    col='surcharge_label',
    col_wrap=2,
    kind='line',
    height=4,
    aspect=1.5,
    linewidth=2,
)

g.set_axis_labels('Chain Step', 'Unweighted TCP Score')
g.set_titles('{col_name}')
g.fig.suptitle('TCP Trajectory Over Chain Steps (Top 3 Strategies)', y=1.05, fontsize=16)
plt.show()

## TCP Distributions By Strategy, Surcharge

we are looking for a strategy with a high median line and a fat upper body, indicating it consistently finds and preserves high-tcp maps.


In [ ]:
CLASS_ORDER = ['Proportional', 'Margin', 'Simple']
valid_classes = [c for c in CLASS_ORDER if c in traces['strategy_class'].unique()]

fig, axes = plt.subplots(1, len(valid_classes), figsize=(7 * len(valid_classes), 7), sharey=True)
if len(valid_classes) == 1:
    axes = [axes]

for i, s_class in enumerate(valid_classes):
    subset = traces[traces['strategy_class'] == s_class].copy()

    ### get strategies in this class, sorted
    strats_in_class = sorted(subset['strategy'].unique())
    subset['strategy'] = pd.Categorical(subset['strategy'], categories=strats_in_class, ordered=True)

    sns.violinplot(
        data=subset,
        x='strategy',
        y=TCP_COL,
        hue='surcharge_label',
        ax=axes[i],
        cut=5,
        inner='quartile',
        linewidth=0.8,
        density_norm='width',
    )

    axes[i].set_title(f'{s_class} Strategies', fontsize=13)
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=30)
    axes[i].grid(True, axis='y', linestyle='--', alpha=0.4)

    ### only show legend on last panel
    if i < len(valid_classes) - 1:
        axes[i].get_legend().remove()
    else:
        axes[i].legend(title='Surcharge', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)

axes[0].set_ylabel('TCP Score')
fig.suptitle('TCP Distribution by Strategy and Surcharge', fontsize=15, y=1.02)
fig.tight_layout()
plt.show()


## Summary Table
aggregated stats per strategy x surcharge combination.


In [ ]:
### mean tcp by strategy x surcharge
pivot = master_df.pivot_table(
    index='strategy',
    columns='surcharge_label',
    values='mean_tcp',
    aggfunc='mean'
).round(4)

print('mean tcp by strategy x surcharge:')
display(pivot)

### same for county splits
pivot_splits = master_df.pivot_table(
    index='strategy',
    columns='surcharge_label',
    values='mean_county_splits',
    aggfunc='mean'
).round(2)

print('\nmean county splits by strategy x surcharge:')
display(pivot_splits)
